# Assignment 3: Classification Models and Evaluation Matrix

## Ecommerce Customer Churn Prediction Using Logistic Regression and SVM

This assignment trains and compares two classification models:

1. Logistic Regression  
2. Support Vector Machine (SVM)

The models are evaluated using:

- Confusion matrix
- Accuracy
- Precision
- Recall
- F1-score

## 1. Import Required Libraries

The required Python libraries are imported first. Pandas and NumPy are used for data handling, while Scikit-learn is used for preprocessing, model training, and evaluation.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## 2. Load the Dataset

The dataset used in this notebook is the Ecommerce Customer Churn Dataset.

In [3]:
df = pd.read_csv("ecommerce_customer_churn_dataset.csv")

df.head()

,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


## 3. Inspect the Dataset

Before building the models, the dataset is inspected to understand the number of rows and columns, data types, and missing values.

In [4]:
print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape:
(325, 14)

Column names:
Index(['Customer_ID', 'Age', 'Gender', 'Region', 'Membership_Type',
       'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase',
       'Customer_Support_Calls', 'Average_Rating', 'Used_Coupon',
       'Newsletter_Subscribed', 'Device_Type', 'Churned'],
      dtype='object')

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               325 non-null    int64  
 1   Age                       320 non-null    float64
 2   Gender                    320 non-null    object 
 3   Region                    320 non-null    object 
 4   Membership_Type           320 non-null    object 
 5   Monthly_Spending          320 non-null    float64
 6   Number_of_Orders          325 non-null    int64  
 7   Days_Since_Last_Purchase  325 non-null    i

## 4. Business Problem

The business problem is to predict whether an ecommerce customer will churn or remain active.

Customer churn means that a customer stops purchasing from or using the ecommerce platform. Predicting churn is useful because the business can identify customers who may leave and take action through retention campaigns, discounts, improved service, or personalized communication.

## 5. Identify Features and Target Variable

The target variable is `Churned`.

The input features are the customer demographic, spending, engagement, and service-related variables.

The column `Customer_ID` is removed because it is only an identifier and does not provide meaningful predictive information.

In [5]:
X = df.drop(columns=["Customer_ID", "Churned"])
y = df["Churned"]

print("Feature dataset shape:", X.shape)
print("Target variable shape:", y.shape)

print("\nTarget variable distribution:")
print(y.value_counts())

Feature dataset shape: (325, 12)
Target variable shape: (325,)

Target variable distribution:
Churned
No     238
Yes     87
Name: count, dtype: int64


## 6. Identify Numerical and Categorical Features

Numerical and categorical columns are separated because they require different preprocessing steps.

In [6]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numerical features:")
print(list(numerical_features))

print("\nCategorical features:")
print(list(categorical_features))

Numerical features:
['Age', 'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase', 'Customer_Support_Calls', 'Average_Rating']

Categorical features:
['Gender', 'Region', 'Membership_Type', 'Used_Coupon', 'Newsletter_Subscribed', 'Device_Type']


## 7. Create Preprocessing Steps

The preprocessing pipeline handles missing values, numerical scaling, and categorical encoding.

For numerical variables:

- Missing values are filled using the median
- Values are standardized using `StandardScaler`

For categorical variables:

- Missing values are filled using the most frequent category
- Categories are converted into numerical format using `OneHotEncoder`

In [10]:
# Preprocessing for numerical columns
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

# Preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)])

## 8. Split the Data into Training and Testing Sets

The dataset is split into training and testing sets.

- 80% of the data is used for training
- 20% of the data is used for testing

The `stratify=y` option is used to keep the same proportion of churn and non-churn customers in both sets.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (260, 12)
X_test shape: (65, 12)
y_train shape: (260,)
y_test shape: (65,)


## 9. Train Model 1: Logistic Regression

Logistic Regression is used as the first classification model. It is a simple and interpretable baseline model for binary classification problems.

In [12]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))])

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

## 10. Evaluate Logistic Regression

The Logistic Regression model is evaluated using confusion matrix, accuracy, precision, recall, and F1-score.

In [13]:
cm_logistic = confusion_matrix(y_test, y_pred_logistic)

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic, pos_label="Yes")
recall_logistic = recall_score(y_test, y_pred_logistic, pos_label="Yes")
f1_logistic = f1_score(y_test, y_pred_logistic, pos_label="Yes")

print("Logistic Regression Confusion Matrix:")
print(cm_logistic)

print("\nLogistic Regression Metrics:")
print("Accuracy:", round(accuracy_logistic, 4))
print("Precision:", round(precision_logistic, 4))
print("Recall:", round(recall_logistic, 4))
print("F1-score:", round(f1_logistic, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

Logistic Regression Confusion Matrix:
[[42  6]
 [ 6 11]]

Logistic Regression Metrics:
Accuracy: 0.8154
Precision: 0.6471
Recall: 0.6471
F1-score: 0.6471

Classification Report:
              precision    recall  f1-score   support

          No       0.88      0.88      0.88        48
         Yes       0.65      0.65      0.65        17

    accuracy                           0.82        65
   macro avg       0.76      0.76      0.76        65
weighted avg       0.82      0.82      0.82        65



## 11. Train Model 2: Support Vector Machine (SVM)

Support Vector Machine is used as the second classification model. SVM can capture more complex decision boundaries than Logistic Regression.

In [16]:
svm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC())])

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

## 12. Evaluate SVM

The SVM model is evaluated using the same metrics so that both models can be compared fairly.

In [17]:
cm_svm = confusion_matrix(y_test, y_pred_svm)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, pos_label="Yes")
recall_svm = recall_score(y_test, y_pred_svm, pos_label="Yes")
f1_svm = f1_score(y_test, y_pred_svm, pos_label="Yes")

print("SVM Confusion Matrix:")
print(cm_svm)

print("\nSVM Metrics:")
print("Accuracy:", round(accuracy_svm, 4))
print("Precision:", round(precision_svm, 4))
print("Recall:", round(recall_svm, 4))
print("F1-score:", round(f1_svm, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

SVM Confusion Matrix:
[[46  2]
 [ 7 10]]

SVM Metrics:
Accuracy: 0.8615
Precision: 0.8333
Recall: 0.5882
F1-score: 0.6897

Classification Report:
              precision    recall  f1-score   support

          No       0.87      0.96      0.91        48
         Yes       0.83      0.59      0.69        17

    accuracy                           0.86        65
   macro avg       0.85      0.77      0.80        65
weighted avg       0.86      0.86      0.85        65



## 13. Compare the Two Models

The results from Logistic Regression and SVM are placed in one comparison table.

In [18]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM"],
    "Accuracy": [accuracy_logistic, accuracy_svm],
    "Precision": [precision_logistic, precision_svm],
    "Recall": [recall_logistic, recall_svm],
    "F1-score": [f1_logistic, f1_svm]})

comparison

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.815385,0.647059,0.647059,0.647059
1,SVM,0.861538,0.833333,0.588235,0.689655


## Task 2: Interpretation and Submission Summary

### Which Model Performed Better?

The better model is the one with the stronger overall evaluation results, especially the higher F1-score and recall. In a customer churn problem, accuracy alone is not enough because the business wants to correctly identify customers who may leave.

If the SVM model has a higher F1-score or recall than Logistic Regression, then SVM performed better overall. If Logistic Regression has similar results, it may still be preferred because it is simpler and easier to interpret.

### Which Metric is Most Important for the Business Problem?

For customer churn prediction, **recall** is one of the most important metrics.

Recall measures how many actual churned customers were correctly identified by the model. This matters because missing customers who are likely to churn can lead to lost revenue. A business would rather identify more at-risk customers early so that it can take retention actions.

### What Do False Positives and False Negatives Mean?

A **false positive** means the model predicts that a customer will churn, but the customer actually stays active. This may cause the business to spend extra money on discounts or retention campaigns for a customer who was not going to leave.

A **false negative** means the model predicts that a customer will stay, but the customer actually churns. This is more serious because the business may lose the customer without taking any action.

### What is One Possible Limitation or Bias in the Model?

One limitation is that the dataset may not include every factor that affects churn. For example, competitor prices, customer complaints, product quality, marketing history, and customer satisfaction may also influence whether a customer leaves.

Another possible issue is class imbalance. If there are more non-churned customers than churned customers, the model may become better at predicting the majority class and weaker at identifying customers who actually churn.

### Why Should Human Judgment Still Be Used?

Human judgment should still be used because machine learning models are based only on the data provided. A model may not understand current business conditions, customer emotions, unusual events, or company strategy.

Business managers and analysts should use model predictions as decision-support tools, not as the only basis for action. Human judgment is needed to decide which customers should receive retention offers and what type of action is most appropriate.

### Final Conclusion

This notebook successfully trained and compared two classification models: Logistic Regression and SVM. Both models were evaluated using confusion matrix, accuracy, precision, recall, and F1-score.

The project shows a complete supervised machine learning workflow, including loading the dataset, understanding the business problem, preprocessing the data, training models, evaluating results, comparing performance, and interpreting the results in a business context
